In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm

from sklearn.cluster import KMeans
import networkx as nx
# import geopandas as gpd

import pandas as pd
import numpy as np


col_names = [
    'Unique_ID', 'CompetitionFY', 'Institution', 'Institution_FR',
    'Total_Amount', 'Program_Name', 'Program_Type', 'Title', 'Main_Discipline', 'Area_of_Research'
]

In [ ]:
U15 = ["University of Alberta", "University of British Columbia", "University of Calgary", "Dalhousie University", "Université Laval", 
       "University of Manitoba", "McGill University", "McMaster University", "Université de Montréal", "University of Ottawa", 
       "Queen's University", "University of Saskatchewan", "University of Toronto", "University of Waterloo", "University of Western Ontario"]

In [ ]:
tri_agency_path = "clean_data/TRIAGENCY_DATA.csv"
TRIAGENCY_DATA = pd.read_csv(tri_agency_path)

TRIAGENCY_DATA.head()

In [ ]:
total_sum = TRIAGENCY_DATA["Total_Amount"].sum()
print("Total Sum: ", total_sum)

for agency in TRIAGENCY_DATA["Agency"].unique():
    print("Total", agency, "Funding: ", TRIAGENCY_DATA[TRIAGENCY_DATA["Agency"] == agency]["Total_Amount"].sum())

In [ ]:
for agency in TRIAGENCY_DATA['Agency'].unique():
    agency_grants = TRIAGENCY_DATA[TRIAGENCY_DATA['Agency'] == agency]
    num_grants = len(agency_grants)
    avg_total_amount = agency_grants['Total_Amount'].mean()

    print(f'Agency: {agency}, Number of Grants: {num_grants}, Average Grant Total Amount: {avg_total_amount}')


In [ ]:
U15_DATA = TRIAGENCY_DATA[TRIAGENCY_DATA["Institution"].isin(U15)]
U15_total_sum = U15_DATA["Total_Amount"].sum()
print("U15 Avg sum: ", U15_total_sum/15)

SFU_DATA = TRIAGENCY_DATA[TRIAGENCY_DATA["Institution"] == "Simon Fraser University"]
sfu_total_sum = SFU_DATA["Total_Amount"].sum()
print("SFU total sum: ", sfu_total_sum)


In [ ]:
# for university in U15:
#     print(f"{university} total %: {U15_DATA[U15_DATA['Institution'] == university]['Total_Amount'].sum()/total_sum * 100:.2f}")

In [ ]:
print("SFU total %: ", sfu_total_sum/total_sum * 100)

**Trend Analysis**

In [ ]:
# Group by year and sum funding amounts
df_yearly = TRIAGENCY_DATA.groupby(TRIAGENCY_DATA['CompetitionFY'])['Total_Amount'].sum().reset_index()

# Plotting the trend
plt.figure(figsize=(10,6))
plt.plot(df_yearly['CompetitionFY'], df_yearly['Total_Amount'], marker='o')
plt.title('Annual Grant Funding Trend')
plt.xlabel('Year')
plt.ylabel('Total Funding Amount')
plt.grid(True)
plt.show()

**Funding Distribution Analysis**

In [ ]:
# sns.barplot(x='Main_Discipline', y='Total_Amount', data=TRIAGENCY_DATA[TRIAGENCY_DATA['Agency'] == 'CIHR'])
# plt.title('Funding Distribution by Research Field')
# plt.xlabel('Research Field')
# plt.ylabel('Total Funding Amount')
# plt.xticks(rotation=90)
# plt.show()
# plt.close()

In [ ]:
data = TRIAGENCY_DATA[TRIAGENCY_DATA['Agency'] == 'CIHR'].groupby('Main_Discipline')['Total_Amount'].sum()

fig, ax = plt.subplots()
colors_mpl = plt.cm.tab20.colors[:len(data.index)]
ax.pie(data, labels=None, colors=colors_mpl,autopct='%1.1f%%', startangle=90, pctdistance=0.85)

# Move the labels outside the pie chart by increasing the labeldistance
plt.legend(data.index, loc="center left", bbox_to_anchor=(1, 0.5))

plt.title('CIHR Funding Distribution')
plt.show()

In [ ]:
data = TRIAGENCY_DATA[TRIAGENCY_DATA['Agency'] == 'NSERC'].groupby('Main_Discipline')['Total_Amount'].sum()

fig, ax = plt.subplots()
colors_mpl = plt.cm.tab20.colors[:len(data.index)]
ax.pie(data, labels=None, colors=colors_mpl,autopct='%1.1f%%', startangle=90, pctdistance=0.85)

# Move the labels outside the pie chart by increasing the labeldistance
plt.legend(data.index, loc="center left", bbox_to_anchor=(1, 0.5))

plt.title('NSERC Funding Distribution')
plt.show()

In [ ]:
# data = U15_DATA[U15_DATA['Agency'] == 'NSERC'].groupby('Main_Discipline')['Total_Amount'].sum()

# fig, ax = plt.subplots()
# colors_mpl = plt.cm.tab20.colors[:len(data.index)]
# ax.pie(data, labels=None, colors=colors_mpl,autopct='%1.1f%%', startangle=90, pctdistance=0.85)

# # Move the labels outside the pie chart by increasing the labeldistance
# plt.legend(data.index, loc="center left", bbox_to_anchor=(1, 0.5))

# plt.title('NSERC Funding Distribution')
# plt.show()

In [ ]:
data = TRIAGENCY_DATA[TRIAGENCY_DATA['Agency'] == 'SSHRC'].groupby('Main_Discipline')['Total_Amount'].sum().nlargest(12)

fig, ax = plt.subplots()
colors_mpl = plt.cm.tab20.colors[:len(data.index)]
ax.pie(data, labels=None, colors=colors_mpl,autopct='%1.1f%%', startangle=90, pctdistance=0.85)

# Move the labels outside the pie chart by increasing the labeldistance
plt.legend(data.index, loc="center left", bbox_to_anchor=(1, 0.5))

plt.title('SSHRC Funding Distribution')
plt.show()

**SFU MARKET SHARE**

In [ ]:
# Example: Group by institution and calculate total funding
# df_institution = TRIAGENCY_DATA.groupby('institution')['funding_amount'].sum().reset_index()

# Plotting SFU’s share vs others
# sfu_funding = df_institution[df_institution['institution'] == 'SFU']
# other_funding = df_institution[df_institution['institution'] != 'SFU']

data = U15_DATA.groupby('Institution')['Total_Amount'].sum()
sfu_total = SFU_DATA['Total_Amount'].sum()
other_totals = data.tolist()
other_labels = data.index.tolist()

# Combine SFU and other institutions' totals and labels
heights = [sfu_total] + other_totals
labels = ['SFU'] + other_labels

colors_mpl = plt.cm.tab20.colors[:len(labels)]

# Plot with dummy x values (e.g., 0, 1, 2, ...)
x = list(range(len(heights)))

for i in range(len(heights)):
    plt.bar(x[i], heights[i], label=labels[i], color=colors_mpl[i])


plt.title("SFU’s Market Share in Grant Funding")
plt.ylabel("Total Funding Amount")
plt.xticks([])  # Hide x-axis tick labels
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(heights, labels=None, colors=colors_mpl, autopct='%1.1f%%', startangle=140)
plt.legend(labels, loc="center left", bbox_to_anchor=(1, 0.5))
plt.title("SFU’s Market Share in Grant Funding")
plt.axis('equal')  # Equal aspect ratio makes the pie a circle
plt.tight_layout()
plt.show()

In [ ]:
sfu_by_year = SFU_DATA.groupby('CompetitionFY')['Total_Amount'].sum()
# data = U15_DATA.groupby('Institution', 'CompetitionFY')['Total_Amount'].sum()
# print(data)
# other_by_year = data.groupby()['Total_Amount'].sum()

# Plotting
plt.figure(figsize=(10, 6))
colors_mpl = plt.cm.tab20.colors[:16]

i = 0
plt.plot(sfu_by_year.index, sfu_by_year.values, label='SFU', marker='o', color=colors_mpl[i])
for institution in U15_DATA["Institution"].unique():
    other_by_year = U15_DATA[U15_DATA["Institution"] == institution].groupby('CompetitionFY')["Total_Amount"].sum()
    i += 1
    plt.plot(other_by_year.index, other_by_year.values, label=institution, marker='o', color=colors_mpl[i])
    

# plt.plot(data.index, data.values, label='Other Institutions', marker='o')

plt.title("Grant Funding Over Time")
plt.xlabel("CompetitionFY")
plt.ylabel("Total Funding Amount")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
sfu_by_year = SFU_DATA.groupby('CompetitionFY')['Total_Amount'].sum()
others_by_year = U15_DATA.groupby('CompetitionFY')['Total_Amount'].mean()
# print(data)
# other_by_year = data.groupby()['Total_Amount'].sum()

# Plotting
plt.figure(figsize=(10, 6))

i = 0
plt.plot(sfu_by_year.index, sfu_by_year.values, label='SFU', marker='o')
plt.plot(other_by_year.index, other_by_year.values, label="u15 mean", marker='o')
    

# plt.plot(data.index, data.values, label='Other Institutions', marker='o')

plt.title("Grant Funding Over Time")
plt.xlabel("CompetitionFY")
plt.ylabel("Total Funding Amount")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**MISCELLANEOUS**

In [ ]:
# tmp_data = TRIAGENCY_DATA.copy()

# tmp_data['Area_of_Research_encoded'] = tmp_data['Area_of_Research'].astype('category').cat.codes

# # Select features for clustering
# X = tmp_data[['Total_Amount', 'Area_of_Research_encoded']]

# # Apply KMeans clustering (e.g., 3 clusters)
# kmeans = KMeans(n_clusters=5)
# tmp_data['cluster'] = kmeans.fit_predict(X)

# # Visualize the clusters
# plt.scatter(tmp_data['Total_Amount'], tmp_data['Area_of_Research_encoded'], c=tmp_data['cluster'], cmap='viridis')
# plt.title('Grant Clusters by Funding and Area of Research')
# plt.xlabel('Funding Amount')
# plt.ylabel('Encoded Area of Research')
# plt.show()


In [ ]:
# # Frequency of funding programs
# funding_program_count = TRIAGENCY_DATA['Program_Name'].value_counts()

# # Plotting the frequency distribution
# funding_program_count.plot(kind='bar', figsize=(10, 6))
# plt.title('Frequency of Grants by Program')
# plt.xlabel('Program Name')
# plt.ylabel('Count of Grants')
# plt.xticks(rotation=90)
# plt.show()


In [ ]:
# df_program_year = TRIAGENCY_DATA.groupby(['CompetitionFY', 'Program_Name'])['Total_Amount'].sum().reset_index()

# # Pivot the data to have programs as columns and years as rows
# df_pivot = df_program_year.pivot(index='CompetitionFY', columns='Program_Name', values='Total_Amount')

# # Plot the trend of each program's funding over time
# df_pivot.plot(figsize=(12, 8))
# plt.title('Grant Funding Trends by Program Over Time')
# plt.xlabel('Year')
# plt.ylabel('Total Funding Amount')
# plt.legend(title='Program Name')
# plt.grid(True)
# plt.show()
